# 数据整合：311 + Crime + ACS + VIIRS

In [3]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import glob
from datetime import datetime
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 配置路径（相对路径）

In [9]:
# 相对路径配置
DATA_DIR = "./finaldata"  # 改这里
PROCESSED_DIR = os.path.join(DATA_DIR, "processed_data")

print(f"数据目录: {os.path.abspath(DATA_DIR)}")
print(f"处理目录: {os.path.abspath(PROCESSED_DIR)}")

数据目录: D:\pen\MUSA5500GeospatialDataScienceInPython\final\finaldata
处理目录: D:\pen\MUSA5500GeospatialDataScienceInPython\final\finaldata\processed_data


## 1. 加载VIIRS数据

In [10]:
# 加载VIIRS结果
viirs_file = os.path.join(PROCESSED_DIR, "philly_lighting_data.shp")
gdf = gpd.read_file(viirs_file)

print(f"✓ VIIRS数据: {len(gdf)} block groups")
print(f"  列数: {len(gdf.columns)}")
gdf.head()

✓ VIIRS数据: 1338 block groups
  列数: 56


,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,GEOID,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,...,viirs_2035,viirs_2036,viirs_2037,viirs_2038,viirs_2039,viirs_2040,viirs_all_,viirs_chan,viirs_pct_,geometry
0,42,101,019000,1,421010190001,Block Group 1,G5030,S,99233,0,...,721.5,716.250000,806.75,869.750000,744.437500,785.187500,797.500000,84.562500,10.769707,"POLYGON ((-75.10314 40.0062, -75.10283 40.0077..."
1,42,101,019100,3,421010191003,Block Group 3,G5030,S,153496,0,...,626.0,643.833333,779.00,712.569444,657.930556,649.472222,670.531532,63.097222,9.715138,"POLYGON ((-75.11181 40.01366, -75.10965 40.013..."
2,42,101,019700,3,421010197003,Block Group 3,G5030,S,58267,0,...,765.0,775.500000,833.50,872.833333,728.875000,745.666667,778.824324,127.166667,17.054067,"POLYGON ((-75.13878 40.01868, -75.13846 40.020..."
3,42,101,019900,1,421010199001,Block Group 1,G5030,S,108795,0,...,816.0,811.500000,881.50,895.666667,753.000000,744.625000,794.148649,151.041667,20.284232,"POLYGON ((-75.14099 40.00553, -75.14083 40.006..."
4,42,101,020000,1,421010200001,Block Group 1,G5030,S,124485,0,...,1354.5,1603.000000,1835.00,1457.625000,1370.791667,1426.791667,1429.418919,30.833333,2.161024,"POLYGON ((-75.15372 39.99799, -75.15368 39.998..."


## 2. 检查数据文件

In [11]:
# 检查所有数据文件夹
data_folders = ['311_service', 'crime', 'acs']

for folder in data_folders:
    folder_path = os.path.join(DATA_DIR, folder)
    if os.path.exists(folder_path):
        files = glob.glob(os.path.join(folder_path, "*"))
        print(f"\n{folder}: {len(files)} 个文件")
        for f in files[:3]:
            size_mb = os.path.getsize(f) / (1024*1024)
            print(f"  {os.path.basename(f)} ({size_mb:.1f} MB)")
    else:
        print(f"\n⚠️ {folder}: 文件夹不存在")


311_service: 3 个文件
  311 Service and Information Requests.csv (105.7 MB)
  311_streetlight_outages.csv (2.2 MB)
  311_streetlight_outages.geojson (5.8 MB)

crime: 4 个文件
  Crime Incidents 2025.csv (36.1 MB)
  crime_2025_night.geojson (27.9 MB)
  crime_2025_night_grid_500m.csv (0.6 MB)

acs: 2 个文件
  acs_philadelphia_tracts.geojson (0.3 MB)
  数据清洗acs.qmd (0.0 MB)


## 3. 处理311数据

**注意：需要先查看数据格式再运行**

In [12]:
# 找311文件
service_311_dir = os.path.join(DATA_DIR, "311_service")
files_311 = glob.glob(os.path.join(service_311_dir, "*.csv"))  # 假设是CSV

if not files_311:
    files_311 = glob.glob(os.path.join(service_311_dir, "*"))  # 所有文件

print(f"311文件: {len(files_311)} 个")
service_311_file = files_311[0] if files_311 else None
print(f"使用: {os.path.basename(service_311_file) if service_311_file else 'None'}")

311文件: 2 个
使用: 311 Service and Information Requests.csv


In [13]:
# 先查看311数据结构
if service_311_file:
    # 尝试读取前几行
    df_311_sample = pd.read_csv(service_311_file, nrows=5, low_memory=False)
    print("311数据列名:")
    print(list(df_311_sample.columns))
    print("\n前5行:")
    print(df_311_sample.head())
else:
    print("❌ 没有找到311数据文件")

311数据列名:
['objectid', 'service_request_id', 'subject', 'status', 'status_notes', 'service_name', 'service_code', 'agency_responsible', 'service_notice', 'requested_datetime', 'updated_datetime', 'expected_datetime', 'closed_datetime', 'address', 'zipcode', 'media_url', 'lat', 'lon']

前5行:
   objectid  service_request_id  \
0   5499388            18676583   
1   5499389            18676584   
2   5499390            18676586   
3   5499391            18676587   
4   5499392            18676588   

                                             subject  status  \
0             Rubbish/Recyclable Material Collection  Closed   
1                     Street Trees (SERVICE REQUEST)  Closed   
2  What numbers do I call to contact PECO and Ver...  Closed   
3         How can I reach Call Before You Dig? (811)  Closed   
4             Rubbish/Recyclable Material Collection    Open   

        status_notes                            service_name service_code  \
0                NaN  Rubbish/Recycla

In [15]:
# 直接用路灯专用文件
streetlight_file = os.path.join(DATA_DIR, "311_service", "311_streetlight_outages.csv")

df_311_lights = pd.read_csv(streetlight_file, low_memory=False)
print(f"路灯311数据: {len(df_311_lights)} 行")

# 查看列名
print(f"列名: {list(df_311_lights.columns[:10])}")
df_311_lights.head()

路灯311数据: 8302 行
列名: ['objectid', 'service_request_id', 'subject', 'status', 'status_notes', 'service_name', 'service_code', 'agency_responsible', 'service_notice', 'requested_datetime']


,objectid,service_request_id,subject,status,status_notes,service_name,service_code,agency_responsible,service_notice,requested_datetime,updated_datetime,expected_datetime,closed_datetime,address,zipcode,media_url,lat,lon,repair_delay_days
0,5057010,17346618,street light outage,closed,NaN,street light outage,SR-ST04,Streets Department,10 Business Days,2025-01-01 14:19:43+00:00,2025-01-29 14:21:50+00:00,2025-01-17 00:00:00+00:00,2025-01-29 14:21:27+00:00,4601 MORRIS ST,19144.0,https://d17aqltn7cihbm.cloudfront.net/uploads/...,40.020641,-75.166743,28.001204
1,5057016,17346624,street light outage,closed,NaN,street light outage,SR-ST04,Streets Department,10 Business Days,2025-01-01 14:26:59+00:00,2025-01-16 20:25:44+00:00,2025-01-17 00:00:00+00:00,2025-01-16 20:25:33+00:00,2516 S 62ND ST,19142.0,https://d17aqltn7cihbm.cloudfront.net/uploads/...,39.924550,-75.225990,15.249005
2,5057055,17346699,street light outage,closed,NaN,street light outage,SR-ST04,Streets Department,10 Business Days,2025-01-01 18:09:06+00:00,2025-01-06 14:41:59+00:00,2025-01-17 00:00:00+00:00,2025-01-06 14:41:41+00:00,2609 BAINBRIDGE ST,19146.0,https://d17aqltn7cihbm.cloudfront.net/uploads/...,39.945075,-75.183636,4.855961
3,5057060,17346704,street light outage,closed,NaN,street light outage,SR-ST04,Streets Department,10 Business Days,2025-01-01 18:17:47+00:00,2025-01-28 22:14:43+00:00,2025-01-17 00:00:00+00:00,2025-01-24 16:40:41+00:00,1930 SANFORD ST,19116.0,NaN,40.100187,-75.015304,22.932569
4,5057085,17346744,street light outage,closed,NaN,street light outage,SR-ST04,Streets Department,10 Business Days,2025-01-01 20:13:45+00:00,2025-10-28 13:35:36+00:00,2025-01-20 00:00:00+00:00,2025-10-28 13:34:43+00:00,BUSTLETON AVE & COTTMAN AVE,19149.0,NaN,40.047714,-75.059899,299.722894


In [16]:
# 转为GeoDataFrame（根据实际经纬度列名修改）
# 常见列名: 'lat', 'lon', 'latitude', 'longitude', 'y', 'x'

# 检查是否有坐标
lat_col = 'lat'  # 根据实际修改
lon_col = 'lon'  # 根据实际修改

if lat_col in df_311_lights.columns and lon_col in df_311_lights.columns:
    # 去除缺失坐标
    df_311_lights = df_311_lights.dropna(subset=[lat_col, lon_col])
    
    df_311_geo = gpd.GeoDataFrame(
        df_311_lights,
        geometry=gpd.points_from_xy(df_311_lights[lon_col], df_311_lights[lat_col]),
        crs='EPSG:4326'
    )
    
    print(f"✓ 转换为GeoDataFrame: {len(df_311_geo)} 条有坐标")
else:
    print(f"❌ 未找到坐标列: {lat_col}, {lon_col}")
    print(f"可用列: {list(df_311_lights.columns)}")

✓ 转换为GeoDataFrame: 8302 条有坐标


In [17]:
# 空间join到block groups
df_311_joined = gpd.sjoin(
    df_311_geo, 
    gdf[['GEOID', 'geometry']], 
    how='left', 
    predicate='within'
)

# 统计每个BG的311数量
bg_311_count = df_311_joined.groupby('GEOID').size().reset_index(name='count_311')

# 合并到主数据
gdf = gdf.merge(bg_311_count, on='GEOID', how='left')
gdf['count_311'] = gdf['count_311'].fillna(0)

print(f"✓ 311数据已整合")
print(f"  有311报告的BG: {(gdf['count_311'] > 0).sum()}")
print(f"  总报告数: {gdf['count_311'].sum():.0f}")

✓ 311数据已整合
  有311报告的BG: 1138
  总报告数: 8296


## 4. 处理Crime数据

In [18]:
# 类似311的处理
crime_dir = os.path.join(DATA_DIR, "crime")
files_crime = glob.glob(os.path.join(crime_dir, "*.csv"))

if not files_crime:
    files_crime = glob.glob(os.path.join(crime_dir, "*"))

print(f"Crime文件: {len(files_crime)} 个")
crime_file = files_crime[0] if files_crime else None

if crime_file:
    # 查看结构
    df_crime_sample = pd.read_csv(crime_file, nrows=5, low_memory=False)
    print("\nCrime数据列名:")
    print(list(df_crime_sample.columns))
else:
    print("❌ 没有找到Crime数据")

Crime文件: 2 个

Crime数据列名:
['the_geom', 'cartodb_id', 'the_geom_webmercator', 'objectid', 'dc_dist', 'psa', 'dispatch_date_time', 'dispatch_date', 'dispatch_time', 'hour', 'dc_key', 'location_block', 'ucr_general', 'text_general_code', 'point_x', 'point_y', 'lat', 'lng']


In [19]:
# 读取并处理crime数据（根据实际列名修改）
if crime_file:
    df_crime = pd.read_csv(crime_file, low_memory=False)
    print(f"Crime数据: {len(df_crime)} 行")
    
    # 找坐标列
    lat_col = 'lat'  # 根据实际修改
    lon_col = 'lng'  # 根据实际修改
    
    if lat_col in df_crime.columns and lon_col in df_crime.columns:
        df_crime = df_crime.dropna(subset=[lat_col, lon_col])
        
        df_crime_geo = gpd.GeoDataFrame(
            df_crime,
            geometry=gpd.points_from_xy(df_crime[lon_col], df_crime[lat_col]),
            crs='EPSG:4326'
        )
        
        df_crime_joined = gpd.sjoin(
            df_crime_geo,
            gdf[['GEOID', 'geometry']],
            how='left',
            predicate='within'
        )
        
        bg_crime_count = df_crime_joined.groupby('GEOID').size().reset_index(name='count_crime')
        
        gdf = gdf.merge(bg_crime_count, on='GEOID', how='left')
        gdf['count_crime'] = gdf['count_crime'].fillna(0)
        
        print(f"✓ Crime数据已整合")
        print(f"  有犯罪记录的BG: {(gdf['count_crime'] > 0).sum()}")
        print(f"  总犯罪数: {gdf['count_crime'].sum():.0f}")

Crime数据: 130589 行
✓ Crime数据已整合
  有犯罪记录的BG: 1337
  总犯罪数: 126713


## 5. 处理ACS数据

In [22]:
# ACS数据 - 读取GeoJSON
acs_dir = os.path.join(DATA_DIR, "acs")
files_acs = glob.glob(os.path.join(acs_dir, "*.geojson"))

print(f"ACS文件: {len(files_acs)} 个")

if files_acs:
    acs_file = files_acs[0]
    print(f"使用: {os.path.basename(acs_file)}")
    
    # 读取GeoJSON
    df_acs = gpd.read_file(acs_file)
    print(f"\nACS数据: {len(df_acs)} 行")
    print(f"列名: {list(df_acs.columns)}")
    print(df_acs.head())
else:
    print("❌ 没有找到ACS GeoJSON文件")

ACS文件: 1 个
使用: acs_philadelphia_tracts.geojson

ACS数据: 408 行
列名: ['GEOID', 'population', 'no_vehicle_total', 'no_vehicle', 'poverty_total', 'poverty', 'median_income', 'poverty_rate', 'no_vehicle_rate', 'geometry']
         GEOID  population  no_vehicle_total  no_vehicle  poverty_total  \
0  42101001500      3251.0            1507.0       543.0         3239.0   
1  42101001800      3300.0            1727.0       339.0         3300.0   
2  42101002802      5720.0            2396.0       549.0         5710.0   
3  42101004001      4029.0            2013.0       473.0         4029.0   
4  42101006300      4415.0            1659.0       495.0         4410.0   

   poverty  median_income  poverty_rate  no_vehicle_rate  \
0    199.0       110859.0      0.061439         0.360319   
1    136.0       114063.0      0.041212         0.196294   
2   1047.0        78871.0      0.183363         0.229132   
3    840.0        61583.0      0.208488         0.234973   
4   1740.0        32347.0      0.3

In [24]:
# 直接合并ACS（已经读取好了）
# ACS数据已经有GEOID列
gdf = gdf.merge(
    df_acs[['GEOID', 'population', 'median_income', 'poverty_rate', 'no_vehicle_rate']], 
    on='GEOID', 
    how='left'
)

print("✓ ACS数据已整合")
print(f"  新增变量: population, median_income, poverty_rate, no_vehicle_rate")
print(f"\n当前总列数: {len(gdf.columns)}")

✓ ACS数据已整合
  新增变量: population, median_income, poverty_rate, no_vehicle_rate

当前总列数: 62


## 6. 计算分析指标

In [26]:
# 计算rates
pop_col = 'population'

if pop_col in gdf.columns:
    gdf['crime_rate'] = (gdf['count_crime'] / (gdf[pop_col] + 1)) * 1000
    gdf['report_311_rate'] = (gdf['count_311'] / (gdf[pop_col] + 1)) * 1000
    print("✓ 计算rates")

# 收入分组 - 用cut而不是qcut
income_col = 'median_income'
if income_col in gdf.columns:
    # 用固定区间而不是quartiles
    gdf['income_group'] = pd.cut(
        gdf[income_col],
        bins=[0, 40000, 60000, 80000, 200000],
        labels=['Low', 'Mid-Low', 'Mid-High', 'High']
    )
    print("✓ 计算收入分组")

print(f"\n当前列数: {len(gdf.columns)}")

✓ 计算rates
✓ 计算收入分组

当前列数: 65


## 7. 数据质量检查

In [27]:
# 检查缺失值
key_cols = ['viirs_2024_mean', 'count_311', 'count_crime']
key_cols = [c for c in key_cols if c in gdf.columns]

print("缺失值统计:")
print(gdf[key_cols].isnull().sum())

print("\n数据摘要:")
print(gdf[key_cols].describe())

缺失值统计:
count_311      0
count_crime    0
dtype: int64

数据摘要:
         count_311  count_crime
count  1338.000000  1338.000000
mean      6.200299    94.703288
std       7.440933    88.793067
min       0.000000     0.000000
25%       1.000000    42.000000
50%       4.000000    69.000000
75%       8.000000   116.000000
max      93.000000   821.000000


## 8. 导出整合数据

In [28]:
# 导出
output_shp = os.path.join(PROCESSED_DIR, "philly_integrated_data.shp")
output_csv = os.path.join(PROCESSED_DIR, "philly_integrated_data.csv")

gdf.to_file(output_shp)
print(f"✓ Shapefile: {output_shp}")

# CSV（不含geometry）
gdf_csv = gdf.copy()
gdf_csv['geometry'] = gdf_csv['geometry'].apply(lambda x: x.wkt if x else None)
gdf_csv.to_csv(output_csv, index=False)
print(f"✓ CSV: {output_csv}")

print(f"\n最终数据:")
print(f"  Block groups: {len(gdf)}")
print(f"  字段数: {len(gdf.columns)}")

✓ Shapefile: ./finaldata\processed_data\philly_integrated_data.shp
✓ CSV: ./finaldata\processed_data\philly_integrated_data.csv

最终数据:
  Block groups: 1338
  字段数: 65
